# 時系列データ解析

## 前処理

### 傾向変動、季節変動、不規則変動の確認

解析の最初のステップは、データを視覚的に調べること。  
データの傾向、季節的な変動、ランダムな変動などの特徴を確認する。  
これにより、データの性質を把握できる。  

これらのトレンド、季節成分、残差は`statsmodels`ライブラリの`seasonal_decompose`を使用して可視化する。

In [1]:
import statsmodels.api as sm
import matplotlib.pyplot as plt

# 時系列データの季節成分を分解して可視化する関数  
def plt_seasonal_decompose(df, col_name, period):
    # df: 時系列データのDataFrame
    # col_name: str 分解対象の列名(時系列データ)  
    # periods: int 季節成分の周期(季節性の周期)  

    # statsmodelsのseasonal_decompose関数を使用して、指定した周期で季節成分を分解
    res = sm.tsa.seasonal_decompose(df[col_name], period=period)

    # 季節成分の分解結果を可視化するためのプロット作成 
    fig = res.plot()

    plt.show()

トレンド推定の方法の1つとして移動平均を取ることも一般的である。

In [2]:
df[move_mean_column_name] = df['column_name'].rolling(100).mean().round(1)

NameError: name 'df' is not defined

rolling()のウィンドウサイズを変更することは。データの平滑化やトレンドの特定に影響を与える可能性がある。  
ウィンドウサイズが小さいと、移動平均より短期的な変動に敏感に反応し、大きいと、長期的なトレンドに焦点を与える。  

### 周期性、時間的な相関の確認

データの周期性や時間的な相関を確認することで、データの性質を理解し、適切なモデリング手法を選択するのに役立つ。  
自己相関関数（ACF）や偏自己相関関数（PACF）を使用して、データの相関構造を調査することが一般的。

### 定常性の確認

時系列データの分析では定常性は重要な使用となる。  

【`定常性`とは】  
●時系列データの基本的な特性の1つで、データの統計的性質が時間に依存しないことを示す。  

●強定常性と弱定常性の2つの概念がある。弱定常性は平均と分散が時間に依存しないことを意味し、強定常性はその他の統計的性質も含む。  

●定常性があるデータは、統計モデルの前提条件を満たし、予測モデルを安定させるために重要。  
非定常なデータでは、トレンドや季節性などが強調され、モデルが不安定になることがある。    

定常性がもたらす影響・・・  
・`予測性能の向上`：  
未来の値を予測するためのモデル作成や性能評価において、より信頼性の高い結果を提供できる。非定常なデータでは、トレンドや季節性が協調され、予測が困難になる。  

・`「見せかけの回帰」の回避`：  
非定常なデータ同士で回帰分析を行うと、全く無関係な変数同士であっても非常に高い相関や統計的有意性が算出されてしまう現象が起きる。  

・`統計モデルの適用`：  
定常性のあるデータに対しては、多くの統計モデルが適用可能。例えば、ARIMA(自己回帰和分移動平均モデル)などの一般的な時系列モデルは定常性を前提としている。  

・`統計的仮定の保持`：  
統計的仮定の1つに、観測データが独立同分布であるという仮定がある。定常性がない場合、この仮定が崩れ、統計的な信頼性は損なわれる可能性がある。  

・`モデルの解釈の容易性`：  
定常性を持つデータのモデルは直感的に理解しやすく、パラメータの解釈が容易になる。  
一方、非定常なデータでは、モデルが複雑になり、パラメータの解釈が難しくなることがある。  

・`信頼性の向上`：  
モデルの予測や推定が安定し、結果のばらつきが少なくなる。

つまり、`データの平均と分散が時間に依存しないことが大切`。  

In [ ]:
# 平均、分散の確認  
print(df[column_name].mean())
print(df[column_name].var())

これだけではよくわからないので、`ADF検定`を行い、定常性があるかを確認する。  

#### ADF検定

ADF検定は、時系列データが「定常」か「非定常（単位根を持つ）」かを統計的に判定する仮説検定手法。  
元となるDF検定を拡張し、誤差項の自己相関（過去の値の影響）を考慮できるようにラグ項を追加したモデルを使用する。

◎ 仮説の設定と判定ルール  
一般的な統計的仮説検定と異なり、`「非定常である（単位根を持つ）（＝時間とともに平均や分散が変化する）」が帰無仮説`に設定される。  
・p値 <= 0.05：帰無仮説を棄却＝データは定常  
・p値 > 0.05：帰無仮説を棄却できない＝データは非定常  

In [3]:
from statsmoels.tsa.stattools import adfuller

def adf_test(series):
    adf_df = pd.DataFrame(
        [adfuller(series)[1]],
        column=["P値"]
    )
    adf_df["P値"] = adf_df["P値"].round(3).astype(str)
    print(adf_df)

adf_test(series=df[column_name])

ModuleNotFoundError: No module named 'statsmoels'

ここで`P値(有意水準)が0.05以下であれば、定常性がある`、といえる。  
0.05より大きければ、非定常なデータであるため、特徴量を追加・生成したり、定常性を持つようにデータ変換する必要がある。  

### （定常性がない場合)データ変換を行い、再度ADF検定を行う

データ変換として一般的なのは、`差分変換`と`対数変換`である。  
・差分変換：1時点離れたデータとの差を取る手法。  
・対数変換：変動が著しく大きな時系列データに対し、原系列の傾向を保持したまま値を小さくする手法。  

データ変換後は、平均と分散がどのように変化しているかを見てみるといい。  
この時、並行して特徴量エンジニアリングをしても良い。  
そして、再度ADF検定を行い、定常性の確認を行う。  

### （予測モデルの作成）

定常性が認められれば、定期せつな予測モデル（ARIMA, Expotential Smoothingなど）を構築し、未来の値を予測するモデルを訓練する。  
（ここでラグ特徴量を取っていけばいい）